In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

print("Debug: Librarie imported succesfully")

filepath="Dataset/Naive-Bayes-Classification-Data.csv"

Debug: Librarie imported succesfully


In [2]:
class PatternClassifiers :

    def __init__(self) :
        self.classes= None
        self.priors={}
        self.mean={}
        self.variance={}


    def displayParam(self):
        print("Prior Probability")
        print(self.priors)
        print("Mean Glucose")
        print(self.mean)
        print("Variance Glucose")
        print(self.variance)


    def bayesianClassifier(self,x, mean, variance) :
        if (variance == 0 ) :
            variance=0.00000001

        exponent= np.exp(-(x-mean)**2) / (variance)
        probability=exponent / np.sqrt(2 * 3.174* variance)

        return probability


    def fit(self, feature, classification) :
        uniqueClass=list(set(classification))
        totalLength = len(classification)

        for c in uniqueClass :
            self.priors[c]=classification.count(c) / totalLength
            feature_c = [feature[i] for i in range(totalLength) if classification[i]==c]

            class_mean = sum(feature_c) / len(feature_c)
            self.mean[c] = class_mean

            class_variance = sum((x - class_mean) **2 for x in feature_c)/len(feature_c)
            self.variance[c] = class_variance

        self.classes = uniqueClass


    def predict_batch(self, feature) :
        prediction=[]
        for x in feature :
            prediction.append(self.predict(x))
        return prediction


    def predict(self, featureValue) :
        probabilities = {}
        for c in self.classes:

            likelihood = self.bayesianClassifier(featureValue, self.mean[c], self.variance[c])
            prior = self.priors[c]

            probabilities[c] = likelihood * prior

        predicted_class = max(probabilities, key=probabilities.get)

        return predicted_class

       


In [3]:
# Loading dataset

dataset=pd.read_csv(filepath)

dataset.info()

dataset.head(15)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 995 entries, 0 to 994
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   glucose        995 non-null    int64
 1   bloodpressure  995 non-null    int64
 2   diabetes       995 non-null    int64
dtypes: int64(3)
memory usage: 23.4 KB


,glucose,bloodpressure,diabetes
0,40,85,0
1,40,92,0
2,45,63,1
3,45,80,0
4,40,73,1
5,45,82,0
6,40,85,0
7,30,63,1
8,65,65,1
9,45,82,0


In [4]:
# Clean mapping of class to diabetes column and dropping the diabetes column

dataset["Class"]=dataset["diabetes"].map({0:"Non-Diabetic",1:"Diabetic"})


GlucoseLevel=dataset["glucose"].to_list()
IsDiabetic=dataset["Class"].to_list()

Glucose_train, Glucose_test, Diabetic_train, Diabetic_test = train_test_split(
    GlucoseLevel,
    IsDiabetic,
    test_size=0.20,
    stratify=IsDiabetic,
    random_state=12
)


In [5]:

classifier = PatternClassifiers()
classifier.fit(Glucose_train, Diabetic_train)


In [6]:
classifier.displayParam()

Prior Probability
{'Non-Diabetic': 0.5, 'Diabetic': 0.5}
Mean Glucose
{'Non-Diabetic': 44.08291457286432, 'Diabetic': 44.2713567839196}
Variance Glucose
{'Non-Diabetic': 13.2921201484811, 'Diabetic': 77.98666700335849}


In [7]:
Diabetic_Pred = classifier.predict_batch((Glucose_test))

print(Diabetic_Pred)


['Non-Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Diabetic', 'Non-Diabetic', 'Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Diabetic', 'Non-Diabetic', 'Diabetic', 'Non-Diabetic', 'Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Non-Diabetic', 'Non-Diabet

In [8]:
correct_predictions = sum(1 for true, pred in zip(Diabetic_test, Diabetic_Pred) if true == pred)


total_predictions = len(Diabetic_test)
accuracy = correct_predictions / total_predictions

print(f"Accuracy: {accuracy * 100:.2f}%\n")

Accuracy: 61.31%



In [9]:
# Confusion Matrix

print("--- Confusion Matrix ---")
conf_matrix = confusion_matrix(Diabetic_test, Diabetic_Pred)
print(conf_matrix, "\n")

--- Confusion Matrix ---
[[24 76]
 [ 1 98]] 



In [10]:
# Classification Report

print("--- Classification Report ---")
class_report = classification_report(Diabetic_test, Diabetic_Pred)
print(class_report)

--- Classification Report ---
              precision    recall  f1-score   support

    Diabetic       0.96      0.24      0.38       100
Non-Diabetic       0.56      0.99      0.72        99

    accuracy                           0.61       199
   macro avg       0.76      0.61      0.55       199
weighted avg       0.76      0.61      0.55       199



In [11]:
# Model Testing

user_input = float(input("Enter a Glucose Value to predict: "))


user_prediction = classifier.predict(user_input)

if user_prediction == 1 or user_prediction == 'Diabetic':
    print("Prediction: Diabetes")
else:
    print("Prediction: No Diabetes")

Prediction: No Diabetes
